# NB08 — Atlas: measure every trained model

        **Up to 45 models · ~25 GPU-hours · shardable across 6 accounts ·
        inference only**


> **New here?** Read `05_PLAIN_ENGLISH_GUIDE.md` first — it explains what this
> project is measuring and why, without jargon. This notebook assumes you have.


        ## What this does

        Exactly what NB02 did, but for the whole atlas instead of four pilot
        models. Per model: attach early-exit points (main network frozen), run
        all 20 compute settings over all 10,000 test images plus a 5,000-image
        training slice, compute the difficulty scores, write the tables.

        ~30–40 minutes per model. Nothing is trained except the small read-off
        heads.

        ## Why it's separate from the training notebooks

        Two reasons. It's cheap and re-runnable, so separating it means a bug in
        the measurement code costs 30 minutes rather than 3 hours. And it can
        start as soon as *any* model finishes training — you don't have to wait
        for the whole atlas.

        ## Safe to re-run

        Models already measured are skipped instantly.

In [ ]:
# === CELL 1 of every notebook: unpack the library ==========================
# This writes two Python files into the session and imports them. Nothing here
# touches the GPU or the network beyond installing three small packages.
#
#   msc_lib   994ee4d87678   the pipeline: HuggingFace sync, model zoo,
#                              measurement, training, the method
#   msc_core  6abdba4ff104   the reference maths: the MSC definition and
#                              every statistic in the paper
#
# Both are generated from KD/src by build_notebooks.py. Editing them HERE does
# nothing useful -- the next rebuild overwrites it. Edit the source instead.
import base64, os, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

# Kaggle images already ship torch, pandas and sklearn. These three vary by
# image version, so we check rather than assume.
#   pyarrow  writes the per-image measurement tables (Parquet)
#   pynvml   reads GPU power/temperature/utilisation directly
#   fvcore   counts FLOPs, which is how compute cost is defined
for _pkg in ('pyarrow', 'pynvml', 'fvcore', 'psutil'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'[BOOT] installing {_pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg,
                        '--break-system-packages'], check=False)

_LIB = (
    '...',
)

_CORE = (
    '...',
)

for _name, _blob in (('msc_lib', _LIB), ('msc_core', _CORE)):
    (WORK / f'{_name}.py').write_bytes(base64.b64decode(''.join(_blob)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
for _m in [m for m in list(sys.modules) if m in ('msc_lib', 'msc_core')]:
    del sys.modules[_m]

import msc_lib as msc
import msc_core

print(f'[BOOT] msc_lib v{msc.__version__} ready  (torch available: {msc._TORCH_OK})')
print(f'[BOOT] artifact space: {msc.WORK_ROOT}   scratch space: {msc.SCRATCH_ROOT}')

## Step 1 — Session

In [ ]:
# === Who am I? =============================================================
#
# THIS NOTEBOOK: 45 run(s), ~27 GPU-hours total.
# At NUM_WORKERS = 1 that is all of it on this account.
# Raise NUM_WORKERS and run the same notebook on each account
# with a different WORKER_ID to divide it.
#
# ACCOUNT   labels this Kaggle account in the run log. Two accounts calling
#           themselves the same thing makes the log useless.
# WORKER_ID splits the work. Every account runs THE SAME notebook; the only
#           thing that differs is this number. Account 1 -> 0, account 2 -> 1,
#           and so on. Each account then works out, by pure arithmetic, which
#           jobs belong to it -- no communication needed, no chance of two
#           accounts training the same model, no chance of a job being missed.
#
# DEFAULT IS 1: this account does everything in this notebook. That is the
# simplest thing that works. Change it only when you actually have several
# accounts running at once.
# Measurement is inference-only: ~30-40 min per model.
ACCOUNT     = 'acct1'      # <<< CHANGE ME
NUM_WORKERS = 1          # <<< how many accounts you are running in parallel
WORKER_ID   = 0            # <<< CHANGE ME: 0, 1, 2, ... up to NUM_WORKERS-1

sess = msc.Session(account=ACCOUNT, phase='p1', dataset='cifar100',
                   worker_id=WORKER_ID, num_workers=NUM_WORKERS,
                   shard_mode='cost',        # balance GPU-hours, not job counts
                   enable_hf=True,
                   session_limit_h=8.5,      # push + pause before Kaggle kills us
                   commits_per_hour_limit=20,# 6 accounts x 20 = 120 < HF's 128/hr
                   batch_interval_sec=1800)  # the 30-minute push policy

## Step 2 — Dataset

In [ ]:
# === Get CIFAR-100 =========================================================
# Looks in this order: attached Kaggle dataset (instant) -> earlier extraction
# -> Kaggle CLI download -> torchvision as a last resort.
# Everything lands in /kaggle/temp (~1 TB scratch), never in /kaggle/working
# (20 GB, and that is the space your results need).
DATA_ROOT = sess.prepare_data()
print('dataset:', DATA_ROOT)

## Step 3 — Catch up

In [ ]:
# === Catch up with what has already been done ==============================
# Downloads only what this notebook needs -- never the whole repo, which would
# fill the 20 GB disk instantly.
#
# It also rebuilds the progress record from the actual training logs instead of
# trusting the status file. If a session died between writing a log and pushing
# its status, those two disagree, and the log is the one that tells the truth.
# Runs marked "finished" that clearly are not get reset so they resume.
sess.sync_state(verbose=True)
sess.status()

## Step 4 — Which models are ready?

Only fully-trained models. A truncated run would produce measurements
from an under-trained network, which are worse than no measurements
because they look valid.

In [ ]:
import pandas as pd
# Identity comes from the run_id, not from ledger fields. Not every event
# carries arch/seed -- repair_ledger reconstructs a completion from history.csv
# and knows only the id -- so reading them from the ledger yields None.
ready = sess.completed_runs(phase='p1')
rdf = pd.DataFrame(ready)
display(rdf)
print(f'\n{len(ready)} trained models available to measure')
if len(rdf):
    print(f"{int(rdf.measured.sum())} already measured, "
          f"{int((~rdf.measured).sum())} remaining")

## Step 5 — Measure this worker's share

In [ ]:
cfgs = []
for r in ready:
    c = sess.config(r['arch'], seed=int(r['seed']))
    c['run_id'] = r['run_id']
    cfgs.append(c)

# stage='measure': the ledger says 'completed' because TRAINING finished, so
# completion for this stage is decided by whether the per-sample tables exist.
results = sess.run_all(cfgs, fn=sess.oracle, title='atlas measurement',
                       done_fn=sess.measured, stage='measure')
for x in results:
    print(f"{x.get('run_id')}: {x.get('status')}")

n_done = sum(1 for c in cfgs if sess.measured(c['run_id']))
print(f'\n{n_done}/{len(cfgs)} of this account\'s runs now measured')

## Step 6 — Row-alignment audit

Every table must describe the same images in the same order. Two tables
that disagree cannot be compared — and comparing them anyway produces
believable nonsense.

In [ ]:
import pandas as pd
rows = []
for d in sorted(sess.runs_dir.iterdir()) if sess.runs_dir.exists() else []:
    ps = d / 'per_sample'
    meta = msc.read_json(ps / 'meta.json', default={}) or {}
    f = ps / 'test.parquet'
    if f.exists():
        df = pd.read_parquet(f, columns=['sample_order_hash'])
        rows.append({'run_id': d.name, 'arch': meta.get('arch'),
                     'rows': len(df),
                     'order_fingerprint': df['sample_order_hash'].iloc[0][:16]})
align = pd.DataFrame(rows)
if not len(align):
    print('No per-sample tables yet -- nothing measured on this account.')
    print('Either the atlas is still training, or another worker owns these runs.')
else:
    display(align)
    k = align.order_fingerprint.nunique()
    print(f'\n{k} distinct ordering(s) across {len(align)} tables -> '
          f"{'OK' if k == 1 else 'MISALIGNED -- STOP AND INVESTIGATE'}")
    msc.save_analysis(sess.data_dir, 'per_sample_alignment', align, sess.hub)

## Step 7 — Finish

In [ ]:
# === Push everything and stop ==============================================
# Blocks until HuggingFace confirms. Safe to re-run.
sess.finish()

# D-19: draining the upload queue is NOT the same as the files being on
# HuggingFace, and "[SESSION] done" reads like a confirmation it is not.
# Ask the repository before you close this tab.
#
# D-20: three states, not two. FINISHED and RESUMABLE are both safe -- a run
# paused at epoch 120 whose ckpt_last.pt is on HF loses nothing when you close
# the tab. Only AT RISK (no summary.json AND no checkpoint) needs action.
try:
    _ids = [c['run_id'] for c in cfgs]
except NameError:
    _ids = []
if _ids:
    sess.confirm_on_hf(_ids)

---
**Next: NB09–NB12**, the analysis. Turn the GPU off for those.